In [2]:
import random
import json
import pandas as pd

# Pools of entity values
equipment = ["brake caliper", "fuel injector", "battery", "engine control unit", "ABS module", "coolant pump", "oxygen sensor", "transmission", "spark plug"]
properties = ["pressure drop", "voltage fluctuation", "temperature spike", "irregular RPM", "overheating", "oil leakage", "fluid contamination"]
faults = ["seized", "short-circuited", "malfunctioned", "failed", "clogged", "worn out", "corroded"]
fault_codes = ["C0050", "C1234", "P0171", "P0300", "P0299", "U0100", "B0028"]
maintenance = ["replaced the unit", "cleaned and reinstalled", "tightened connector", "flushed system", "reset module", "lubricated the parts", "calibrated sensors"]
locations = ["engine bay", "rear suspension", "front axle", "transmission housing", "dashboard area", "under the hood"]
measurements = ["12.6V", "3.2V", "150 PSI", "0.5 bar", "85°C", "1200 RPM"]
vehicles = ["BMW 320i", "Tesla Model 3", "Ford F-150", "Audi A4", "Toyota Camry"]
time_durations = ["within 2 hours", "over 3 days", "after 500 miles", "during startup", "while idling"]
customer_symptoms = ["dashboard warning", "strange noise", "loss of power", "engine stalling", "unusual vibration"]

templates = [
    "The {EN} {FT} due to {DP} (code {FC}) in the {LOC}. The {VH} showed {MV}. Issue: {CS}. We {MM}.",
    "Customer reported {CS} with the {EN}. Fault was {FT} caused by {DP} (fault code {FC}) located at the {LOC}. Maintenance performed: {MM}.",
    "Reported {FT} of the {EN} due to {DP}, confirmed by code {FC} near the {LOC}. Vehicle {VH} readings: {MV}. We {MM}.",
    "The {EN} in the {LOC} {FT}, showing fault code {FC} linked to {DP}. Observed {CS} on the {VH}. Maintenance action: {MM}.",
    "Issue detected {TD} - {EN} {FT} because of {DP} (code {FC}) at the {LOC}. Symptoms: {CS}. We {MM}."
]

def inject_noise(text, level="high"):
    chars = list(text)
    if not chars:
        return text
    if level == "high":
        ops = ['swap', 'delete', 'insert', 'caseflip']
        for _ in range(random.randint(1, 3)):
            op = random.choice(ops)
            idx = random.randint(0, len(chars) - 1)
            if op == 'swap' and idx < len(chars) - 1:
                chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
            elif op == 'delete':
                del chars[idx]
            elif op == 'insert':
                chars.insert(idx, random.choice("abcdefghijklmnopqrstuvwxyz"))
            elif op == 'caseflip':
                chars[idx] = chars[idx].upper() if chars[idx].islower() else chars[idx].lower()
    return ''.join(chars)

def corrupt_entity(entity_value):
    p = random.random()
    if p < 0.1:
        return ""  # drop entity
    elif p < 0.4:
        return inject_noise(entity_value, level="high")
    elif p < 0.6:
        return entity_value.lower()
    elif p < 0.8:
        return inject_noise(entity_value, level="low")
    else:
        return entity_value  # leave as is

def generate_noisy_data(n=2000):
    data = []
    for _ in range(n):
        EN = corrupt_entity(random.choice(equipment))
        DP = corrupt_entity(random.choice(properties))
        FT = corrupt_entity(random.choice(faults))
        FC = corrupt_entity(random.choice(fault_codes))
        MM = corrupt_entity(random.choice(maintenance))
        LOC = corrupt_entity(random.choice(locations))
        MV = corrupt_entity(random.choice(measurements))
        VH = corrupt_entity(random.choice(vehicles))
        TD = corrupt_entity(random.choice(time_durations))
        CS = corrupt_entity(random.choice(customer_symptoms))

        text = random.choice(templates).format(
            EN=EN, DP=DP, FT=FT, FC=FC, MM=MM,
            LOC=LOC, MV=MV, VH=VH, TD=TD, CS=CS
        )

        entities = []
        for label, value in [
            ("EquipmentName", EN),
            ("DeviceProperties", DP),
            ("FaultType", FT),
            ("FaultCode", FC),
            ("MaintenanceMethod", MM),
            ("VehicleComponentLocation", LOC),
            ("MeasurementValue", MV),
            ("VehicleMakeModel", VH),
            ("TimeDuration", TD),
            ("CustomerReportedSymptom", CS)
        ]:
            if value.strip() == "":
                continue
            start = text.find(value)
            if start >= 0:
                entities.append({
                    "start": start,
                    "end": start + len(value),
                    "label": label
                })

        data.append({"text": text, "entities": entities})
    return data

# Create dataset
noisy_dataset = generate_noisy_data(2000)

# Optional: Convert to DataFrame for inspection
df_noisy = pd.DataFrame(noisy_dataset)
print(df_noisy.head(2))

# Save to JSON
with open("noisy_dataset_high_corruption.json", "w", encoding="utf-8") as f:
     json.dump(noisy_dataset, f, indent=2, ensure_ascii=False)


                                                text  \
0  The oxygeN seNor in the  , showing fault code ...   
1  The oxygen sensor in the engine bay short-circ...   

                                            entities  
0  [{'start': 4, 'end': 16, 'label': 'EquipmentNa...  
1  [{'start': 4, 'end': 17, 'label': 'EquipmentNa...  


In [18]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification
from datasets import Dataset
from seqeval.metrics import classification_report
import json

# 1. Load tokenizer and fine-tuned model
model_path = "ner-roberta-vehicle/checkpoint-5000"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# 2. Load noisy dataset
with open("noisy_dataset_high_corruption.json") as f:
    noisy_data = json.load(f)

# 3. Define label list and mappings
label_list = [
    'O', 'B-EquipmentName', 'I-EquipmentName', 'B-DeviceProperties', 'I-DeviceProperties',
    'B-FaultType', 'I-FaultType', 'B-FaultCode', 'I-FaultCode', 'B-MaintenanceMethod', 'I-MaintenanceMethod',
    'B-VehicleComponentLocation', 'I-VehicleComponentLocation', 'B-MeasurementValue', 'I-MeasurementValue',
    'B-VehicleMakeModel', 'I-VehicleMakeModel', 'B-TimeDuration', 'I-TimeDuration',
    'B-CustomerReportedSymptom', 'I-CustomerReportedSymptom'
]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# 4. Tokenize and align labels
def tokenize_and_align(example):
    text = example["text"]
    entities = example.get("entities", [])

    tokenized_input = tokenizer(
        text,
        truncation=True,
        padding='max_length',  # pad sequences to max_length here
        max_length=256,
        return_offsets_mapping=True,
    )

    labels = ["O"] * len(tokenized_input["input_ids"])
    offset_mapping = tokenized_input["offset_mapping"]

    for entity in entities:
        start, end, label = entity["start"], entity["end"], entity["label"]
        entity_label = f"B-{label}"
        inside_label = f"I-{label}"

        for idx, (offset_start, offset_end) in enumerate(offset_mapping):
            # Skip special tokens with offset (0,0)
            if offset_start == 0 and offset_end == 0:
                continue

            if offset_start == start:
                labels[idx] = entity_label
            elif offset_start > start and offset_end <= end:
                labels[idx] = inside_label

    label_ids = [label2id.get(label, 0) for label in labels]
    tokenized_input["labels"] = label_ids
    tokenized_input.pop("offset_mapping")
    return tokenized_input

# 5. Prepare Dataset and tokenize
noisy_dataset = Dataset.from_list(noisy_data)
tokenized_noisy_dataset = noisy_dataset.map(tokenize_and_align, batched=False)

# 6. Data collator and DataLoader
data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)

test_dataloader = DataLoader(tokenized_noisy_dataset, batch_size=8, collate_fn=data_collator)

# 7. Prediction and evaluation
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        for i in range(len(labels)):
            pred_tags = []
            true_tags = []
            for j in range(len(labels[i])):
                if attention_mask[i][j] == 1:
                    true_tag = id2label[labels[i][j].item()]
                    pred_tag = id2label[preds[i][j].item()]
                    true_tags.append(true_tag)
                    pred_tags.append(pred_tag)
            true_labels.append(true_tags)
            predictions.append(pred_tags)

# 8. Print classification report
print(classification_report(true_labels, predictions))


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`text` in this case) have excessive nesting (inputs type `list` where type `int` is expected).